In [11]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/br/qhn11dt91tvcg9cqgczhjdlr0000gn/T/pip-build-env-t6s68if4/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider remo

In [12]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

In [13]:
SEED = 128
df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")

In [14]:
df["settlement_date"] = pd.to_datetime(df["settlement_date"])
df["month"] = df["settlement_date"].dt.month
df["day_of_week"] = df["settlement_date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Capacity utilization ratios
df["wind_utilization"] = df["embedded_wind_generation"] / (df["embedded_wind_capacity"] + 1)
df["solar_utilization"] = df["embedded_solar_generation"] / (df["embedded_solar_capacity"] + 1)

# Net cross-border flow sum
flow_cols = ["ifa2_flow","britned_flow","moyle_flow","east_west_flow","nemo_flow"]
df["net_crossborder_flow"] = df[flow_cols].sum(axis=1)

In [15]:
df_sample = df.sample(50000, random_state=SEED)  # 50k rijen

train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED, 
)

reg = setup(
    data=train_df,
    target="england_wales_demand",
    session_id=SEED,
    fold=2,
    verbose=True,

    transform_target=True,
    remove_multicollinearity=True,
)

best_model = compare_models(sort="MAE", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="MAE", 
    fold=5,
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

save_model(final_model, "../Model_ElecDemand/england_wales_demand_predictor")

,Description,Value
0,Session id,128
1,Target,england_wales_demand
2,Target type,Regression
3,Original data shape,"(35000, 21)"
4,Transformed data shape,"(35000, 21)"
5,Transformed train set shape,"(24500, 21)"
6,Transformed test set shape,"(10500, 21)"
7,Numeric features,19
8,Date features,1
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,980.4501,1803158.9716,1342.7765,0.9653,0.0413,0.0305,0.6450
et,Extra Trees Regressor,1017.2410,2122734.2143,1456.7390,0.9592,0.0456,0.0321,0.6200
rf,Random Forest Regressor,1100.9851,2511658.5421,1584.8129,0.9517,0.0497,0.0348,0.5850
gbr,Gradient Boosting Regressor,1405.8666,3571354.8127,1889.7746,0.9313,0.0574,0.0436,1.1100
dt,Decision Tree Regressor,1559.9185,5263162.4531,2294.1549,0.8988,0.0717,0.0492,0.0550
ada,AdaBoost Regressor,2587.0222,10180809.9315,3190.5478,0.8043,0.1012,0.0834,0.3000
knn,K Neighbors Regressor,2973.9314,15247517.2908,3904.6632,0.7068,0.1214,0.0942,0.1850
lr,Linear Regression,3643.8410,21032241.5342,4586.0227,0.5956,0.1441,0.1162,0.8700
ridge,Ridge Regression,3656.0060,21312345.7194,4616.4661,0.5902,0.1448,0.1164,0.4550
br,Bayesian Ridge,3656.6132,21315316.1513,4616.7871,0.5902,0.1448,0.1164,0.4150


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,752.6692,1071247.2530,1035.0108,0.9797,0.0314,0.0232
1,747.7962,1050636.2219,1025.0055,0.9797,0.0319,0.0233
2,749.7969,1088121.6690,1043.1307,0.9789,0.0328,0.0236
3,747.3453,1055041.7236,1027.1522,0.9798,0.0321,0.0234
4,774.9720,1149153.7357,1071.9859,0.9780,0.0330,0.0241
Mean,754.5159,1082840.1206,1040.4570,0.9792,0.0322,0.0235
Std,10.3991,35692.0911,17.0097,0.0007,0.0006,0.0003


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,670.8451,857650.5966,926.0943,0.9838,0.0287,0.0210


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('target_transformation',
                  TransformerWrapperWithInverse(transformer=TargetTransformer(estimator=PowerTransformer(standardize=False)))),
                 ('date_feature_extractor',
                  TransformerWrapper(include=['settlement_date'],
                                     transformer=ExtractDateTimeFeatures())),
                 ('numerical_imputer',
                  TransformerWrapper(include=['settleme...
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('remove_multicollinearity',
                  TransformerWrapper(exclude=[],
                                     transformer=RemoveMulticollinearity(threshold=0.9))),
                 ('actual_estimator',
                  LGBMRegressor(bagging_fraction=1.0, bagging_freq=3,
                                feature_fraction=0.6, learning_rate=0.2,
                                min_child_sa